In [23]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "password"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))
df.columns = df.columns.str.strip()
# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


# Dashboard Layout / View
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([

    # LOGO AND TITLE
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width':'200px'}
        ),
        html.H1("CS-340 Grazioso Salvare Dashboard", style={'textAlign':'center'}),
        html.H3("Dashboard by Roberto Castro", style={'textAlign':'center'})
    ], style={'textAlign':'center'}),

    html.Hr(),

    # --- FILTER CONTROLS ---
    html.Div([
        html.Label("Rescue Filter:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'}
            ],
            value='reset',
            inline=True
        )
    ], style={'textAlign':'center'}),

    html.Hr(),

    # DATA TABLE
    dash_table.DataTable(
    id='datatable-id',
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} 
              for i in df.columns],
    data=df.to_dict('records'),
    page_size=10,
    sort_action="native",
    filter_action="native",
    style_table={'overflowX': 'auto'},
    row_selectable="single",
    selected_rows=[0],
),

    html.Br(),
    html.Hr(),

    # GRAPHS & MAP
    html.Div(
        style={
            'display': 'flex',
            'flexDirection': 'row',
            'width': '100%',
        },
        children=[
            html.Div(
                id='graph-id',
                style={
                    'flex': '1',
                    'minWidth': '0',
                    'height': '500px',
                    'padding': '0 10px 0 0',
                }
            ),

            html.Div(
                id='map-id',
                style={
                    'flex': '1',
                    'minWidth': '0',
                    'height': '500px',
                    'border': '1px solid #ccc',
                }
            ),
        ]
    ),

])

# Interaction Between Components / Controller
def get_rescue_query(filter_type):
    if filter_type == "water":
        return {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        return {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                "Siberian Husky", "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        return {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher", "German Shepherd", "Golden Retriever",
                "Bloodhound", "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    return {}


    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)
    query = get_rescue_query(filter_type)
    data = db.read(query)
    df = pd.DataFrame.from_records(data)
    
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records')
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return [html.H3("No data available")]

    dff = pd.DataFrame.from_dict(viewData)

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution for Selected Filter'
    )

    return [dcc.Graph(figure=fig)]
    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [{
        'if': { 'column_id': col },
        'background_color': '#D2F3FF'
    } for col in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, selected_rows):

    if not viewData:
        return [html.Div("No data available", style={"color": "red"})]

    dff = pd.DataFrame(viewData)
    dff.columns = dff.columns.str.strip()

    # Pick the selected row, or default to the first one
    row = selected_rows[0] if selected_rows else 0

    try:
        lat = float(dff.loc[row, "location_lat"])
        lon = float(dff.loc[row, "location_long"])
    except:
        return [html.Div("Missing coordinates", style={"color": "red"})]

    name = dff.loc[row].get("name", "Unknown")
    breed = dff.loc[row].get("breed", "Unknown")

    return [
        dl.Map(
            center=[lat, lon],
            zoom=11,
            style={
                'width': '100%',
                'height': '100%',
                'border': '1px solid #ccc'
            },
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(name)
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Connected successfully!
Dash app running on https://swedensharp-subjecttibet-3000.codio.io/proxy/8050/
